In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PROCESSED = Path("../data/processed")

In [2]:
kandy = pd.read_csv(
    DATA_PROCESSED / "training_table_kandy_clean.csv"
)

nuwara_eliya = pd.read_csv(
    DATA_PROCESSED / "training_table_nuwara_eliya.csv"
)

kegalle = pd.read_csv(
    DATA_PROCESSED / "training_table_kegalle.csv"
)

badulla = pd.read_csv(
    DATA_PROCESSED / "training_table_badulla.csv"
)

print("Kandy:", kandy.shape)
print("Nuwara Eliya:", nuwara_eliya.shape)
print("Kegalle:", kegalle.shape)
print("Badulla:", badulla.shape)

Kandy: (577, 17)
Nuwara Eliya: (448, 17)
Kegalle: (146, 17)
Badulla: (292, 17)


In [3]:
tables = {
    "Kandy": kandy,
    "Nuwara Eliya": nuwara_eliya,
    "Kegalle": kegalle,
    "Badulla": badulla
}

for name, df in tables.items():
    print("\n", name)
    print(df.columns.tolist())


 Kandy
['label', 'country', 'source_dataset', 'district', 'slope', 'aspect', 'twi', 'spi', 'dist_to_river', 'dist_to_road', 'elev', 'plan_curv', 'profile_curv', 'x', 'y', 'longitude', 'latitude']

 Nuwara Eliya
['label', 'country', 'source_dataset', 'district', 'slope', 'aspect', 'twi', 'spi', 'dist_to_river', 'dist_to_road', 'elev', 'plan_curv', 'profile_curv', 'x', 'y', 'longitude', 'latitude']

 Kegalle
['label', 'country', 'source_dataset', 'district', 'slope', 'aspect', 'twi', 'spi', 'dist_to_river', 'dist_to_road', 'elev', 'plan_curv', 'profile_curv', 'x', 'y', 'longitude', 'latitude']

 Badulla
['label', 'country', 'source_dataset', 'district', 'slope', 'aspect', 'twi', 'spi', 'dist_to_river', 'dist_to_road', 'elev', 'plan_curv', 'profile_curv', 'x', 'y', 'longitude', 'latitude']


In [4]:
common_cols = set.intersection(
    *[set(df.columns) for df in tables.values()]
)

print("Common columns:")
print(sorted(common_cols))

for name, df in tables.items():
    extra = set(df.columns) - common_cols

    if extra:
        print(
            f"{name} extra columns:",
            extra
        )

Common columns:
['aspect', 'country', 'dist_to_river', 'dist_to_road', 'district', 'elev', 'label', 'latitude', 'longitude', 'plan_curv', 'profile_curv', 'slope', 'source_dataset', 'spi', 'twi', 'x', 'y']


In [5]:
combined_sl = pd.concat(
    [
        kandy,
        nuwara_eliya,
        kegalle,
        badulla
    ],
    ignore_index=True
)

print(
    "Combined shape:",
    combined_sl.shape
)

Combined shape: (1463, 17)


In [6]:
print(
    combined_sl["district"]
    .value_counts()
)

district
Kandy           577
Nuwara Eliya    448
Badulla         292
Kegalle         146
Name: count, dtype: int64


In [7]:
print(
    combined_sl.groupby(
        ["district", "label"]
    )
    .size()
    .unstack(fill_value=0)
)

label           0    1
district              
Badulla       146  146
Kandy         288  289
Kegalle        73   73
Nuwara Eliya  224  224


In [8]:
print(
    combined_sl.isna()
    .sum()
    .sort_values(ascending=False)
)

label             0
country           0
source_dataset    0
district          0
slope             0
aspect            0
twi               0
spi               0
dist_to_river     0
dist_to_road      0
elev              0
plan_curv         0
profile_curv      0
x                 0
y                 0
longitude         0
latitude          0
dtype: int64


In [9]:
print(
    combined_sl[
        ["longitude", "latitude"]
    ].describe()
)

         longitude     latitude
count  1463.000000  1463.000000
mean     80.753901     7.123841
std       0.207178     0.189507
min      80.169529     6.589746
25%      80.616036     6.976546
50%      80.785802     7.139002
75%      80.900390     7.267791
max      81.239705     7.577596


In [10]:
base_path = (
    DATA_PROCESSED /
    "training_table_sri_lanka_selected_districts_base.csv"
)

combined_sl.to_csv(
    base_path,
    index=False
)

print("Saved:", base_path)

Saved: ..\data\processed\training_table_sri_lanka_selected_districts_base.csv


### Adding Rainfall 

In [11]:
# Cell 1 — imports
from pathlib import Path

import numpy as np
from PIL import Image
from scipy import ndimage
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import matplotlib.pyplot as plt

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
DATA_RAW.mkdir(parents=True, exist_ok=True)

In [12]:
# Cell 2 — load your points and reference date
gdf = pd.read_csv(DATA_PROCESSED / "training_table_sri_lanka_selected_districts_base.csv")
gdf = gpd.GeoDataFrame(
    gdf,
    geometry=[Point(xy) for xy in zip(gdf["longitude"], gdf["latitude"])],
    crs="EPSG:4326",
)

REFERENCE_DATE = "2025-12-02"
WINDOWS_DAYS = [1, 3, 7, 15, 30]

In [13]:
# Cell 3 — Open-Meteo fetch function (cumulative totals, not averages)
import time
import requests

API_URL = "https://archive-api.open-meteo.com/v1/archive"

def fetch_daily_precip(lat, lon, start_date, end_date):
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "Asia/Colombo",
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()["daily"]["precipitation_sum"]

In [15]:
# Cell 4 — loop over points, cache by rounded coord, compute window totals
max_window = max(WINDOWS_DAYS)
start_date = (pd.Timestamp(REFERENCE_DATE) - pd.Timedelta(days=max_window - 1)).strftime("%Y-%m-%d")

cache = {}
rain_cols = {f"rain_{d}d": [] for d in WINDOWS_DAYS}

for i, row in gdf.iterrows():
    key = (round(row["latitude"], 3), round(row["longitude"], 3))

    if key not in cache:
        try:
            cache[key] = fetch_daily_precip(*key, start_date, REFERENCE_DATE)
        except Exception as e:
            print(f"Row {i}: {e}")
            cache[key] = [None] * max_window
        time.sleep(0.2)

    daily = cache[key]
    for d in WINDOWS_DAYS:
        vals = [v for v in daily[-d:] if v is not None]
        rain_cols[f"rain_{d}d"].append(sum(vals) if vals else None)

    if i % 100 == 0:
        print(f"{i}/{len(gdf)}")

for col, vals in rain_cols.items():
    gdf[col] = vals

0/1463
100/1463
Row 195: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=30)
200/1463
300/1463
400/1463
500/1463
600/1463
700/1463
800/1463
900/1463
1000/1463
1100/1463
1200/1463
1300/1463
1400/1463


In [16]:
# Cell 5 — save
out_path = DATA_PROCESSED / "training_table_with_rainfall.csv"
gdf.drop(columns="geometry").to_csv(out_path, index=False)
print(f"Saved {out_path}")

Saved ..\data\processed\training_table_with_rainfall.csv


In [6]:
import pandas as pd

merge_df= pd.read_csv(r"D:\PROJECTS\Landslide_prediction\data\processed\training_table_sri_lanka_selected_districts_base.csv")
print(merge_df.head(2))

   label    country            source_dataset district      slope      aspect  \
0      1  Sri Lanka  NBRO_Kandy_MapExtraction    Kandy  25.469864  139.800827   
1      1  Sri Lanka  NBRO_Kandy_MapExtraction    Kandy   5.432209   66.889214   

         twi         spi  dist_to_river  dist_to_road         elev  plan_curv  \
0   1.840257    1.428990    1140.782471   1744.375610  1492.568115  -0.002050   
1  10.333244  277.962769     579.203247    524.562744   275.105865  -0.019511   

   profile_curv              x              y  longitude  latitude  
0     -0.002289  470242.439942  825802.746638  80.730303  7.470790  
1     -0.002210  484953.620429  824300.136779  80.863636  7.457259  
